# JSBrother — Kaggle/Colab GPU LLM Server

Serves **Qwen2.5-7B-Instruct** in 4-bit (`bitsandbytes`) via FastAPI and ngrok.

### Fixes & Improvements Applied:
1. **Decodes only newly generated tokens** (does not echo the input prompt back).
2. **Credentials loaded securely** from Kaggle Secrets (`kaggle_secrets`), Colab secrets (`google.colab.userdata`), or environment variables (no hard-coded tokens).
3. **Qwen2.5-7B-Instruct 4-bit** with chat template (`apply_chat_template`), with `GET /health` and `POST /generate`.


In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn pyngrok nest-asyncio transformers accelerate bitsandbytes torch pydantic


In [ ]:
# 2. Configure credentials securely (No hard-coded secrets)
import os

NGROK_AUTHTOKEN = None
LLM_API_KEY = None

# Attempt 1: Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    NGROK_AUTHTOKEN = user_secrets.get_secret('NGROK_AUTHTOKEN')
    LLM_API_KEY = user_secrets.get_secret('LLM_API_KEY')
except Exception:
    pass

# Attempt 2: Google Colab Userdata Secrets
if not NGROK_AUTHTOKEN or not LLM_API_KEY:
    try:
        from google.colab import userdata
        NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
        LLM_API_KEY = userdata.get('LLM_API_KEY')
    except Exception:
        pass

# Attempt 3: OS Environment variables
if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = os.environ.get('NGROK_AUTHTOKEN', '')
if not LLM_API_KEY:
    LLM_API_KEY = os.environ.get('LLM_API_KEY', 'jsbrother-secret-key')

if not NGROK_AUTHTOKEN:
    print('⚠️ WARNING: NGROK_AUTHTOKEN is missing. Please set NGROK_AUTHTOKEN in Kaggle/Colab Secrets or env.')
else:
    print('✅ NGROK_AUTHTOKEN configured from secrets.')
print(f'✅ API_KEY configured securely.')


In [ ]:
# 3. Load Qwen2.5-7B-Instruct in 4-bit with BitsAndBytesConfig
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = 'Qwen/Qwen2.5-7B-Instruct'
print(f'Loading {model_id} in 4-bit quantization on GPU...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True
)
print('✅ Model and tokenizer loaded successfully!')


In [ ]:
# 4. Define FastAPI app with /health and /generate endpoints
from typing import Optional
from fastapi import FastAPI, Header, HTTPException, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title='JSBrother LLM Server', version='1.0.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class GenerateRequest(BaseModel):
    prompt: str
    system: Optional[str] = None
    max_new_tokens: int = 512
    temperature: float = 0.0

class GenerateResponse(BaseModel):
    text: str

def verify_api_key(authorization: Optional[str] = Header(None)):
    if not authorization:
        raise HTTPException(status_code=401, detail='Missing Authorization header')
    expected = f'Bearer {LLM_API_KEY}'
    if authorization.strip() != expected.strip():
        raise HTTPException(status_code=403, detail='Invalid API Key')

@app.get('/health')
def health_check():
    return {'status': 'ok', 'model': model_id}

@app.post('/generate', response_model=GenerateResponse, dependencies=[Depends(verify_api_key)])
def generate_endpoint(req: GenerateRequest):
    try:
        messages = []
        if req.system:
            messages.append({'role': 'system', 'content': req.system})
        messages.append({'role': 'user', 'content': req.prompt})

        # Apply Qwen chat template
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer([formatted_prompt], return_tensors='pt').to(model.device)
        prompt_token_count = inputs.input_ids.shape[1]

        with torch.no_grad():
            gen_kwargs = {
                'max_new_tokens': req.max_new_tokens,
                'pad_token_id': tokenizer.eos_token_id,
            }
            if req.temperature > 0:
                gen_kwargs['temperature'] = req.temperature
                gen_kwargs['do_sample'] = True
            else:
                gen_kwargs['do_sample'] = False

            output_ids = model.generate(**inputs, **gen_kwargs)

        # Fix (a): Decode ONLY new tokens, NOT the prompt tokens
        new_tokens = output_ids[0][prompt_token_count:]
        generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        return GenerateResponse(text=generated_text)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [ ]:
# 5. Start ngrok tunnel and serve with Uvicorn
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print('=' * 65)
print(f'🚀 JSBrother LLM Server is online!')
print(f'🔗 Public URL:  {public_url}')
print(f'🩺 Health check: {public_url}/health')
print(f'⚙️  Set in JSBrother .env: LLM_BASE_URL={public_url}')
print('=' * 65)

uvicorn.run(app, host='0.0.0.0', port=8000)
